In [ ]:
!python --version

# Split Frames Notebook
 - import packages
 - set footage directory - denote that some clips have been reserved, moved to a test clip dir manually
 - set output frame directory - planning to dump all frames into same dir vs organizing by clip
     - I'll use frame naming to track clip lineage
 - list all files in footage dir
 - loop through files
 - use ffmpeg in a subprocess to break clips into 1 frame / second
 - ???
 - profit ie notebook finished with frames in frames dir

In [ ]:
# imports
import subprocess
from pathlib import Path
from IPython.display import display, Image
from tqdm.notebook import tqdm

In [ ]:
# setup
footage_path = Path("footage")
frames_path = Path("frames")

# some clips have been pulled out for later testing
test_path = Path("testclips")

# how many frames per second to extract
framerate = 1

if not frames_path.exists():
    frames_path.mkdir()
    print(f'Created frames directory: {frames_path}')
else:
    print(f'Frames directory exists: {frames_path}')
    if any(frames_path.iterdir()):
        print(f'Warning: Frames directory is not empty!')

print(f'Looking in footage directory: {footage_path}')
files = sorted(footage_path.glob('*.h264'))
print(f'Found {len(files)} files')

print(f'Looking in test clips directory: {test_path}')
test_files = sorted(test_path.glob('*.h264'))
print(f'Found {len(test_files)} files')
if len(test_files) == 0:
    print('Test clips directory is empty, did you forget to move clips?')

In [ ]:
# test on one file first
file = files[0]

filename = frames_path / f"{file.stem}_%04d.jpg"
framerate = 1

cmd = [
    "ffmpeg",
    "-y",
    "-i", str(file),
    "-vf", f"fps={framerate}",
    "-loglevel", "error",
    str(filename)
]

try:
    result = subprocess.run(cmd, check=True, capture_output=True)
    total_frames = len(list(frames_path.glob(f'{file.stem}*')))
    print(f'{file} processed successfully, created {total_frames} frames')
    first_frame = sorted(frames_path.glob(f'{file.stem}*.jpg'))[0]
    print(f'Displaying first frame: {first_frame}')
    display(Image(filename=str(first_frame), width=600))
except subprocess.CalledProcessError as e:
    print(f'Error on file: {file}')
    print(e)

In [ ]:
for file in tqdm(files):
    filename = frames_path / f"{file.stem}_%04d.jpg"

    if len(list(frames_path.glob(f'{file.stem}*'))) > 0:
        print(f'File {file} already has frames, skipping')
        continue
    
    cmd = [
        "ffmpeg",
        "-y",
        "-i", str(file),
        "-vf", f"fps={framerate}",
        "-loglevel", "error",
        str(filename)
    ]
    
    try:
        result = subprocess.run(cmd, check=True, capture_output=True)
        total_frames = len(list(frames_path.glob(f'{file.stem}*')))
        print(f'{file} processed successfully, created {total_frames} frames')
        first_frame = sorted(frames_path.glob(f'{file.stem}*.jpg'))[0]
        print(f'Displaying first frame: {first_frame}')
        display(Image(filename=str(first_frame), width=600))
    except subprocess.CalledProcessError as e:
        print(f'Error on file: {file}')
        print(e)
        total_frames = len(list(frames_path.glob(f'{file.stem}*')))
        print(f'{file} processed partially, created {total_frames} frames')

total_frames = len(list(frames_path.glob(f'*.jpg')))
print(f'Created {total_frames} frames from {len(files)} files')